# Exhaustive Model Selection (All Combinations)
Evaluate all feature combinations using R², AIC, and BIC.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import itertools

from src.data_prep import prepare_data
from src.features import run_feature_engineering

In [ ]:
df = prepare_data()
df = run_feature_engineering(df)

features = [
    'price','freight_value','delivery_time','lag_price','lag_sales',
    'discount_ratio','month','day_of_week'
]

X_full = df[features].fillna(0)
y = df['profit']

## Run All Combinations

In [ ]:
results = []

for k in range(1, len(features)+1):
    for combo in itertools.combinations(features, k):
        X = sm.add_constant(X_full[list(combo)])
        model = sm.OLS(y, X).fit()

        results.append({
            'features': combo,
            'n_features': len(combo),
            'r2': model.rsquared,
            'adj_r2': model.rsquared_adj,
            'AIC': model.aic,
            'BIC': model.bic
        })

results_df = pd.DataFrame(results)

## Best Models by Metric

In [ ]:
best_r2 = results_df.sort_values('r2', ascending=False).head(10)
best_aic = results_df.sort_values('AIC').head(10)
best_bic = results_df.sort_values('BIC').head(10)

best_r2, best_aic, best_bic

## Interpretation Guide
- R²: explanatory power (higher is better)
- Adjusted R²: penalizes extra variables
- AIC/BIC: penalize complexity (lower is better)

## Optional: Filter Pareto Optimal Models

In [ ]:
pareto = results_df.sort_values(['AIC','BIC','adj_r2'], ascending=[True,True,False]).head(20)
pareto